# Raw 데이터 적재

이 파일은 이동전화 유동인구 데이터와 카드 결제 데이터를 MySQL의 데이터베이스에 적재하기 위한 파일입니다.

원본 데이터를 최대한 그대로 보존하면서 분석 및 전처리 과정에서 원본 데이터의 출처를 추적할 수 있도록 적재 과정을 구성하였습니다.

## 주요 내용
- 환경변수를 활용하여 MySQL 연결
- 프로젝트 내부 원본 데이터 파일 탐색
- 원본 데이터 컬럼 구조 검증
- 이동전화 유동인구 데이터 Raw 적재
- 카드 결제 데이터 Raw 적재
- `source_file`, `source_row_num`을 활용하여 원본 추적
- 파일별 중복 적재 방지
- 적재 결과 검증

## 결과
- `bigcontest_raw.flow_age` 적재
- `bigcontest_raw.flow_time` 적재
- `bigcontest_raw.flow_wkdy` 적재
- `bigcontest_raw.card_topic1` 적재
- 원본 파일별 데이터 행 수 적재 상태 확인

## 1. Imports & Function Definition

In [24]:
import os
import pandas as pd
import unicodedata

from pathlib import Path
from dotenv import load_dotenv

import pymysql
from pymysql.constants import CLIENT

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [27]:
def normalize_filename(filename):
    """
    Windows/macOS 간 한글 파일명 Unicode 표현 차이를 통일합니다.
    """
    return unicodedata.normalize("NFC", filename)
    

def find_project_root(start=None):
    """
    .env 파일을 기준으로 프로젝트의 최상위 경로를 탐색합니다.

    args:
        - start: 프로젝트 루트 탐색을 시작할 경로 (None인 경우 현재 작업 경로에서 탐색을 시작)

    return:
        - .env 파일이 존재하는 프로젝트 루트의 Path 객체
    """
    # 현재 작업 경로 또는 지정된 경로에서 탐색 시작
    current = Path(start or Path.cwd()).resolve()

    # 상위 디렉터리로 이동하면서 .env 파일 탐색
    while current != current.parent:
        if (current / ".env").exists():
            return current

        current = current.parent

    # 최상위 경로까지 탐색했지만 .env 파일이 없는 경우 예외 처리
    raise FileNotFoundError(
        "프로젝트 루트의 .env 파일을 찾을 수 없습니다."
    )


def execute_sql_file(sql_file, host, port, user, password):
    """
    SQL 파일을 읽어 MySQL 서버에서 실행합니다.

    args:
        - sql_file: 실행할 SQL 파일의 Path 객체
        - host: MySQL 서버 주소
        - port: MySQL 서버 포트
        - user: MySQL 사용자명
        - password: MySQL 비밀번호

    return:
        - 반환값 없음
    """
    # SQL 파일 읽기
    sql_script = sql_file.read_text(encoding="utf-8")

    # 데이터베이스를 지정하지 않고 MySQL 서버에 연결
    connection = pymysql.connect(
        host=host,
        port=port,
        user=user,
        password=password,
        charset="utf8mb4",
        autocommit=True,
        client_flag=CLIENT.MULTI_STATEMENTS
    )

    try:
        with connection.cursor() as cursor:
            # SQL 파일 전체 실행
            cursor.execute(sql_script)

            # SHOW TABLES, DESC 등 여러 결과 집합 순차 처리
            while cursor.nextset():
                pass

    finally:
        connection.close()


def validate_columns(df, expected_columns, file_name):
    """
    실제 데이터의 컬럼 구조가 정의서의 예상 컬럼 구조와 일치하는지 검증합니다.

    args:
        - df: 컬럼 구조를 검증할 DataFrame
        - expected_columns: 정의서 기준 예상 컬럼 목록
        - file_name: 검증 대상 원본 파일명

    return:
        - 반환값 없음
        - 컬럼 구조가 다를 경우 ValueError 발생
    """
    # 실제 데이터의 컬럼 목록 추출
    actual_columns = list(df.columns)

    # 컬럼명과 컬럼 순서가 예상 구조와 동일한지 확인
    if actual_columns != expected_columns:
        raise ValueError(
            f"""
            컬럼 구조가 정의서와 다릅니다.
            
            파일:
            {file_name}
            
            예상 컬럼:
            {expected_columns}
            
            실제 컬럼:
            {actual_columns}
            """
        )


def is_file_loaded(engine, table_name, source_file):
    """
    특정 원본 파일이 RAW 테이블에 이미 적재되어 있는지 확인합니다.

    args:
        - engine: MySQL 연결에 사용할 SQLAlchemy Engine 객체
        - table_name: 적재 여부를 확인할 RAW 테이블명
        - source_file: 적재 여부를 확인할 원본 파일명

    return:
        - 이미 적재된 경우 True
        - 적재되지 않은 경우 False
    """
    # source_file 기준으로 기존 적재 행 수 조회
    query = text(f"""
        SELECT COUNT(*)
        FROM {table_name}
        WHERE source_file = :source_file
    """)

    with engine.connect() as conn:
        count = conn.execute(
            query,
            {"source_file": source_file}
        ).scalar()

    return count > 0


def load_flow_file(
    engine,
    file_path,
    table_name,
    expected_columns,
    chunksize=5000
):
    """
    유동인구 원본 파일을 읽어 지정한 MySQL RAW 테이블에 적재합니다.

    args:
        - engine: MySQL 연결에 사용할 SQLAlchemy Engine 객체
        - file_path: 적재할 유동인구 원본 파일의 Path 객체
        - table_name: 데이터를 적재할 RAW 테이블명
        - expected_columns: 정의서 기준 예상 컬럼 목록
        - chunksize: MySQL에 한 번에 적재할 행의 개수

    return:
        - 반환값 없음
        - 파일이 이미 적재된 경우 적재를 수행하지 않음
    """
    source_file = normalize_filename(file_path.name)

    # 동일한 원본 파일의 중복 적재 방지
    if is_file_loaded(engine, table_name, source_file):
        print(
            f"[SKIP] {source_file} "
            f"→ 이미 {table_name}에 존재합니다."
        )
        return

    print(f"[READ] {source_file}")

    # 유동인구 CSV는 파이프(|)를 구분자로 사용
    # 코드성 컬럼은 문자열로 읽어 pandas의 자동 타입 추론을 방지
    df = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        sep="|",
        dtype={
            "STD_YM": "string",
            "BLOCK_CD": "string"
        }
    )

    # 컬럼명에 존재할 수 있는 앞뒤 공백 제거
    df.columns = df.columns.str.strip()

    # 실제 컬럼 구조와 정의서 기준 컬럼 구조 비교
    validate_columns(
        df,
        expected_columns,
        source_file
    )

    # 원본 파일 추적을 위한 메타데이터 추가
    df["source_file"] = source_file

    # 원본 파일 내 데이터 행 번호 기록
    df["source_row_num"] = range(
        1,
        len(df) + 1
    )

    print(
        f"[LOAD] {source_file}"
        f" → {table_name}"
        f" ({len(df):,} rows)"
    )

    # MySQL RAW 테이블에 데이터 추가
    # raw_id와 loaded_at은 MySQL에서 자동 생성
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=chunksize,
        method="multi"
    )

    print(
        f"[DONE] {source_file}"
        f" → {len(df):,} rows\n"
    )


def load_card_file(
    engine,
    file_path,
    expected_columns,
    table_name="card_topic1",
    read_chunksize=50_000,
    insert_chunksize=2_000
):
    """
    카드 결제 원본 파일을 chunk 단위로 읽어 MySQL RAW 테이블에 적재합니다.

    args:
        - engine: MySQL 연결에 사용할 SQLAlchemy Engine 객체
        - file_path: 적재할 카드 원본 파일의 Path 객체
        - expected_columns: 정의서 기준 예상 컬럼 목록
        - table_name: 데이터를 적재할 RAW 테이블명
        - read_chunksize: 원본 파일을 한 번에 읽을 행의 개수
        - insert_chunksize: MySQL에 한 번에 적재할 행의 개수

    return:
        - 반환값 없음
        - 파일이 이미 적재된 경우 적재를 수행하지 않음
    """
    source_file = normalize_filename(file_path.name)

    # 동일한 원본 파일의 중복 적재 방지
    if is_file_loaded(engine, table_name, source_file):
        print(
            f"[SKIP] {source_file} "
            f"→ 이미 {table_name}에 존재합니다."
        )
        return

    print(f"[LOAD START] {source_file}")

    total_rows = 0
    source_row_num = 1

    # 전체 파일 적재를 하나의 transaction으로 처리
    with engine.begin() as conn:

        # 대용량 카드 데이터를 일정 행 단위로 나누어 읽기
        for chunk_no, chunk in enumerate(
            pd.read_csv(
                file_path,
                sep="\t",
                encoding="cp949",
                dtype=str,
                keep_default_na=False,
                chunksize=read_chunksize
            ),
            start=1
        ):
            # 컬럼명 앞뒤 공백 제거
            chunk.columns = chunk.columns.str.strip()

            # 실제 컬럼 구조와 정의서 기준 컬럼 구조 비교
            validate_columns(
                chunk,
                expected_columns,
                source_file
            )

            # 원본 파일 추적을 위한 메타데이터 추가
            chunk["source_file"] = source_file

            # 전체 파일을 기준으로 원본 행 번호 기록
            chunk["source_row_num"] = range(
                source_row_num,
                source_row_num + len(chunk)
            )

            # 현재 chunk를 MySQL RAW 테이블에 추가
            chunk.to_sql(
                name=table_name,
                con=conn,
                if_exists="append",
                index=False,
                chunksize=insert_chunksize,
                method="multi"
            )

            # 누적 적재 행 수 및 다음 시작 행 번호 갱신
            total_rows += len(chunk)
            source_row_num += len(chunk)

            print(
                f"[CHUNK {chunk_no:02d}] "
                f"{total_rows:,} rows 적재 완료"
            )

    print()
    print(
        f"[DONE] {source_file} "
        f"→ 총 {total_rows:,} rows"
    )

---

## 2. 프로젝트 환경 설정 및 Raw 데이터베이스 생성

프로젝트 루트 경로를 설정하고 Raw 데이터베이스 생성용 SQL 파일을 실행합니다.

SQL 파일을 통해 `bigcontest_raw` 데이터베이스와 Raw 데이터 적재에 필요한 테이블을 생성한 뒤,

SQLAlchemy를 이용하여 생성된 데이터베이스에 연결합니다.

### 주요 작업
- 프로젝트 루트 경로 탐색
- `.env` 환경변수 로드
- Raw 데이터베이스 생성 SQL tlfgod
- `bigcontest_raw` 데이터베이스 연결
- 생성된 Raw 테이블 확인

In [3]:
# 프로젝트 루트 경로 탐색
PROJECT_ROOT = find_project_root()

print("PROJECT_ROOT:", PROJECT_ROOT)

# .env 파일의 환경변수 로드
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ROOT: /Users/lee-jongyoon/Documents/bigcontest


True

In [4]:
# Raw 데이터베이스 및 테이블 생성 SQL 파일 경로
RAW_SQL_FILE = PROJECT_ROOT / "sql" / "raw_create_table.sql"

# SQL 파일 실행
execute_sql_file(
    sql_file=RAW_SQL_FILE,
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

print("RAW 데이터베이스 및 테이블 생성 완료")

RAW 데이터베이스 및 테이블 생성 완료


In [5]:
# 생성된 bigcontest_raw 데이터베이스에 연결할 URL 생성
db_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
    query={"charset": "utf8mb4"}
)

# SQLAlchemy Engine 생성
engine = create_engine(
    db_url,
    pool_pre_ping=True
)

In [6]:
# 연결된 데이터베이스 확인
with engine.connect() as conn:
    database = conn.execute(
        text("SELECT DATABASE();")
    ).scalar()

print("Connected Database:", database)

Connected Database: bigcontest_raw


In [7]:
# 생성된 Raw 테이블 확인
pd.read_sql(
    """
    SHOW TABLES;
    """,
    engine
)

,Tables_in_bigcontest_raw
0,card_topic1
1,flow_age
2,flow_time
3,flow_wkdy


## 3. 원본 데이터 파일 탐색 및 컬럼 구조 정의

프로젝트 디렉터리에서 이동전화 유동인구 데이터와 카드 결제 원본 파일을 자동으로 탐색합니다.

또한 데이터 정의서를 기준으로 각 데이터셋의 예상 컬럼 구조를 정의하여,
이후 Raw 적재 과정에서 실제 파일의 컬럼 구조가 올바른지 검증할 수 있도록 준비합니다.

### 주요 작업
- 유동인구 파일 자동 탐색
- 카드 데이터 파일 자동 탐색
- 탐색된 파일 개수 및 파일명 확인
- 데이터셋별 예상 컬럼 목록 정의

In [26]:
# 프로젝트 디렉터리에서 데이터 종류별 원본 파일 탐색

age_files = sorted(
    PROJECT_ROOT.rglob("flow_age_pop_*.csv")
)

time_files = sorted(
    PROJECT_ROOT.rglob("flow_time_pop_*.csv")
)

wkdy_files = sorted(
    PROJECT_ROOT.rglob("flow_wkdy_pop_*.csv")
)


# 카드 데이터 파일명
CARD_FILENAME = "신한카드_빅콘테스트2026_데이터1.txt"

# Windows/macOS 한글 파일명 호환을 위해 Unicode 정규화 후 비교
card_files = [
    path
    for path in PROJECT_ROOT.rglob("*.txt")
    if normalize_filename(path.name)
    == normalize_filename(CARD_FILENAME)
]

In [9]:
# 탐색된 파일 개수와 파일명 확인
print("AGE:", len(age_files))
for f in age_files:
    print(f.name)

print()

print("TIME:", len(time_files))
for f in time_files:
    print(f.name)

print()

print("WKDY:", len(wkdy_files))
for f in wkdy_files:
    print(f.name)

print()

print("CARD:", len(card_files))
for f in card_files:
    print(f.name)

AGE: 6
flow_age_pop_202507.csv
flow_age_pop_202508.csv
flow_age_pop_202509.csv
flow_age_pop_202510.csv
flow_age_pop_202511.csv
flow_age_pop_202512.csv

TIME: 6
flow_time_pop_202507.csv
flow_time_pop_202508.csv
flow_time_pop_202509.csv
flow_time_pop_202510.csv
flow_time_pop_202511.csv
flow_time_pop_202512.csv

WKDY: 6
flow_wkdy_pop_202507.csv
flow_wkdy_pop_202508.csv
flow_wkdy_pop_202509.csv
flow_wkdy_pop_202510.csv
flow_wkdy_pop_202511.csv
flow_wkdy_pop_202512.csv

CARD: 0


In [10]:
# 성별, 연령별 유동인구 컬럼 정의
FLOW_AGE_COLUMNS = [
    "STD_YM",
    "BLOCK_CD",
    "X_COORD",
    "Y_COORD",
    "MAN_FLOW_POP_CNT_10G",
    "MAN_FLOW_POP_CNT_20G",
    "MAN_FLOW_POP_CNT_30G",
    "MAN_FLOW_POP_CNT_40G",
    "MAN_FLOW_POP_CNT_50G",
    "MAN_FLOW_POP_CNT_60GU",
    "WMAN_FLOW_POP_CNT_10G",
    "WMAN_FLOW_POP_CNT_20G",
    "WMAN_FLOW_POP_CNT_30G",
    "WMAN_FLOW_POP_CNT_40G",
    "WMAN_FLOW_POP_CNT_50G",
    "WMAN_FLOW_POP_CNT_60GU"
]

# 시간대별 유동인구 컬럼 정의
FLOW_TIME_COLUMNS = [
    "STD_YM",
    "BLOCK_CD",
    "X_COORD",
    "Y_COORD"
] + [
    f"TMST_{hour:02d}"
    for hour in range(24)
]

# 요일별 유동인구 컬럼 정의
FLOW_WKDY_COLUMNS = [
    "STD_YM",
    "BLOCK_CD",
    "X_COORD",
    "Y_COORD",
    "FLOW_POP_CNT_MON",
    "FLOW_POP_CNT_TUS",
    "FLOW_POP_CNT_WED",
    "FLOW_POP_CNT_THU",
    "FLOW_POP_CNT_FRI",
    "FLOW_POP_CNT_SAT",
    "FLOW_POP_CNT_SUN"
]

# 카드 결제 데이터 컬럼 정의
CARD_COLUMNS = [
    "TA_YMD",
    "TIME_GB",
    "MCT_SGG_CD",
    "MCT_RY_CD",
    "SEX_CCD",
    "AGE_CCD",
    "TS_AT",
    "USE_CNT"
]

## 4. 이동전화 유동인구 Raw 적재

탐색한 이동전화 유동인구 원본 파일을 `bigcontest_raw` 데이터베이스의 각 Raw 테이블에 적재합니다.

유동인구 데이터는 데이터 특성에 따라 성·연령별, 시간대별, 요일별 데이터로 구분하여 적재합니다.

이미 적재된 원본 파일은 `source_file`을 기준으로 확인하여 중복 적재하지 않고 건너뜁니다.

### 주요 작업
- 성·연령별 유동인구 데이터 적재
- 시간대별 유동인구 데이터 적재
- 요일별 유동인구 데이터 적재
- 원본 파일 단위 중복 적재 방지

#### 성·연령별 유동 인구

월별 성·연령별 유동인구 원본 파일을 `flow_age` 테이블에 적재합니다.

In [11]:
# 성·연령별 유동인구 파일을 순차적으로 Raw 테이블에 적재
for file_path in age_files:
    load_flow_file(
        engine=engine,
        file_path=file_path,
        table_name="flow_age",
        expected_columns=FLOW_AGE_COLUMNS
    )

[READ] flow_age_pop_202507.csv
[LOAD] flow_age_pop_202507.csv → flow_age (98,129 rows)
[DONE] flow_age_pop_202507.csv → 98,129 rows

[READ] flow_age_pop_202508.csv
[LOAD] flow_age_pop_202508.csv → flow_age (99,115 rows)
[DONE] flow_age_pop_202508.csv → 99,115 rows

[READ] flow_age_pop_202509.csv
[LOAD] flow_age_pop_202509.csv → flow_age (99,469 rows)
[DONE] flow_age_pop_202509.csv → 99,469 rows

[READ] flow_age_pop_202510.csv
[LOAD] flow_age_pop_202510.csv → flow_age (100,750 rows)
[DONE] flow_age_pop_202510.csv → 100,750 rows

[READ] flow_age_pop_202511.csv
[LOAD] flow_age_pop_202511.csv → flow_age (100,444 rows)
[DONE] flow_age_pop_202511.csv → 100,444 rows

[READ] flow_age_pop_202512.csv
[LOAD] flow_age_pop_202512.csv → flow_age (98,345 rows)
[DONE] flow_age_pop_202512.csv → 98,345 rows



#### 시간대별 유동인구 적재

월별 시간대별 유동인구 원본 파일을 `flow_time` 테이블에 적재합니다.

참고: 원본 데이터에 중복이 존재하지만, Raw 단계에서는 제거하지 않고 원본을 보존합니다.

In [12]:
# 시간대별 유동인구 파일을 순차적으로 Raw 테이블에 적재
for file_path in time_files:
    load_flow_file(
        engine=engine,
        file_path=file_path,
        table_name="flow_time",
        expected_columns=FLOW_TIME_COLUMNS
    )

[READ] flow_time_pop_202507.csv
[LOAD] flow_time_pop_202507.csv → flow_time (94,099 rows)
[DONE] flow_time_pop_202507.csv → 94,099 rows

[READ] flow_time_pop_202508.csv
[LOAD] flow_time_pop_202508.csv → flow_time (95,374 rows)
[DONE] flow_time_pop_202508.csv → 95,374 rows

[READ] flow_time_pop_202509.csv
[LOAD] flow_time_pop_202509.csv → flow_time (95,084 rows)
[DONE] flow_time_pop_202509.csv → 95,084 rows

[READ] flow_time_pop_202510.csv
[LOAD] flow_time_pop_202510.csv → flow_time (97,881 rows)
[DONE] flow_time_pop_202510.csv → 97,881 rows

[READ] flow_time_pop_202511.csv
[LOAD] flow_time_pop_202511.csv → flow_time (95,822 rows)
[DONE] flow_time_pop_202511.csv → 95,822 rows

[READ] flow_time_pop_202512.csv
[LOAD] flow_time_pop_202512.csv → flow_time (185,090 rows)
[DONE] flow_time_pop_202512.csv → 185,090 rows



#### 요일별 유동인구

월별 요일별 유동인구 원본 파일을 `flow_wkdy` 테이블에 적재합니다.

In [13]:
# 요일별 유동인구 파일을 순차적으로 RAW 테이블에 적재
for file_path in wkdy_files:
    load_flow_file(
        engine=engine,
        file_path=file_path,
        table_name="flow_wkdy",
        expected_columns=FLOW_WKDY_COLUMNS
    )

[READ] flow_wkdy_pop_202507.csv
[LOAD] flow_wkdy_pop_202507.csv → flow_wkdy (117,825 rows)
[DONE] flow_wkdy_pop_202507.csv → 117,825 rows

[READ] flow_wkdy_pop_202508.csv
[LOAD] flow_wkdy_pop_202508.csv → flow_wkdy (118,958 rows)
[DONE] flow_wkdy_pop_202508.csv → 118,958 rows

[READ] flow_wkdy_pop_202509.csv
[LOAD] flow_wkdy_pop_202509.csv → flow_wkdy (119,865 rows)
[DONE] flow_wkdy_pop_202509.csv → 119,865 rows

[READ] flow_wkdy_pop_202510.csv
[LOAD] flow_wkdy_pop_202510.csv → flow_wkdy (120,643 rows)
[DONE] flow_wkdy_pop_202510.csv → 120,643 rows

[READ] flow_wkdy_pop_202511.csv
[LOAD] flow_wkdy_pop_202511.csv → flow_wkdy (122,248 rows)
[DONE] flow_wkdy_pop_202511.csv → 122,248 rows

[READ] flow_wkdy_pop_202512.csv
[LOAD] flow_wkdy_pop_202512.csv → flow_wkdy (242,534 rows)
[DONE] flow_wkdy_pop_202512.csv → 242,534 rows



## 5. 카드 결제 데이터 Raw 적재

신한카드 원본 데이터를 `bigcontest_raw.card_topic1` 테이블에 적재합니다.

카드 데이터는 약 100만 행 규모의 대용량 TXT 파일이므로 메모리 사용량을 줄이기 위해 일정 행 단위로 나누어 읽고 적재합니다.

또한 카드 원본 파일은 `CP949` 인코딩과 탭(`\t`) 구분자를 사용하므로 해당 형식에 맞게 데이터를 읽습니다.

이미 적재된 원본 파일은 `source_file`을 기준으로 확인하여 중복 적재하지 않고 건너뜁니다.

### 주요 작업
- 카드 원본 파일 존재 여부 확인
- 카드 데이터 샘플 확인
- 카드 데이터 RAW 적재
- chunk 단위 대용량 데이터 처리

#### 카드 원본 파일 확인

프로젝트에서 탐색된 카드 데이터 파일이 정확히 1개인지 확인하고 적재 대상 파일을 저장합니다.

In [28]:
# 카드 데이터 파일이 정확히 1개 존재하는지 확인
if len(card_files) != 1:
    raise ValueError(
        f"카드 데이터1 파일이 {len(card_files)}개 발견되었습니다."
    )

# 적재 대상 카드 파일 지정
card_file = card_files[0]

print("CARD FILE:", card_file.name)

CARD FILE: 신한카드_빅콘테스트2026_데이터1.txt


#### 카드 데이터 샘플 확인
전체 데이터를 적재하기 전에 일부 행을 읽어 인코딩, 구분자 및 컬럼 구조가 정상적으로 인식되는지 확인합니다.

In [29]:
# 전체 적재 전 카드 데이터 일부 행 확인
test_card = pd.read_csv(
    card_file,
    sep="\t",
    encoding="cp949",
    dtype=str,
    keep_default_na=False,
    nrows=5
)

test_card

,TA_YMD,TIME_GB,MCT_SGG_CD,MCT_RY_CD,SEX_CCD,AGE_CCD,TS_AT,USE_CNT
0,20250704,12_17,서울 강남구,가구,법인,법인,15477163,16
1,20250721,12_17,서울 강남구,가구,법인,법인,308451,38
2,20250703,12_17,서울 강남구,가구,여성,20 대,78970,5
3,20250703,18_23,서울 강남구,가구,남성,20 대,271220,5
4,20250706,18_23,서울 강남구,가구,여성,20 대,260872,5


In [30]:
# 카드 데이터 컬럼 구조 확인
print(test_card.columns.tolist())
print("컬럼 수:", len(test_card.columns))

['TA_YMD', 'TIME_GB', 'MCT_SGG_CD', 'MCT_RY_CD', 'SEX_CCD', 'AGE_CCD', 'TS_AT', 'USE_CNT']
컬럼 수: 8


#### 카드 데이터 Raw 적재

카드 원본 데이터를 일정 크기의 chunk 단위로 나누어 `card_topic1` 테이블에 적재합니다.

In [31]:
# 카드 결제 데이터를 RAW 테이블에 적재
load_card_file(
    engine=engine,
    file_path=card_file,
    expected_columns=CARD_COLUMNS,
    table_name="card_topic1"
)

[LOAD START] 신한카드_빅콘테스트2026_데이터1.txt
[CHUNK 01] 50,000 rows 적재 완료
[CHUNK 02] 100,000 rows 적재 완료
[CHUNK 03] 150,000 rows 적재 완료
[CHUNK 04] 200,000 rows 적재 완료
[CHUNK 05] 250,000 rows 적재 완료
[CHUNK 06] 300,000 rows 적재 완료
[CHUNK 07] 350,000 rows 적재 완료
[CHUNK 08] 400,000 rows 적재 완료
[CHUNK 09] 450,000 rows 적재 완료
[CHUNK 10] 500,000 rows 적재 완료
[CHUNK 11] 550,000 rows 적재 완료
[CHUNK 12] 600,000 rows 적재 완료
[CHUNK 13] 650,000 rows 적재 완료
[CHUNK 14] 700,000 rows 적재 완료
[CHUNK 15] 750,000 rows 적재 완료
[CHUNK 16] 800,000 rows 적재 완료
[CHUNK 17] 850,000 rows 적재 완료
[CHUNK 18] 900,000 rows 적재 완료
[CHUNK 19] 950,000 rows 적재 완료
[CHUNK 20] 1,000,000 rows 적재 완료
[CHUNK 21] 1,044,710 rows 적재 완료

[DONE] 신한카드_빅콘테스트2026_데이터1.txt → 총 1,044,710 rows


## 6. RAW 적재 결과 검증

이동전화 유동인구 데이터와 카드 결제 데이터가 MySQL Raw 테이블에 정상적으로 적재되었는지 확인합니다.

Raw 단계에서는 데이터의 정제나 이상값 처리를 수행하지 않고,
테이블별 적재 행 수, 원본 파일별 적재 현황 및 원본 행 번호의 연속성을 검증합니다.

### 주요 작업
- RAW 테이블별 전체 적재 행 수 확인
- 원본 파일별 적재 행 수 확인
- `source_row_num`의 시작값, 종료값 및 중복 여부 확인
- Raw 데이터 샘플 확인

#### 테이블별 전체 적재 행 수 확인
각 Raw 테이블에 적재된 전체 데이터 행 수를 확인합니다.

In [32]:
# RAW 테이블별 전체 적재 행 수 확인
query = """
SELECT  'flow_age' AS table_name
        ,COUNT(*) AS row_count
  FROM  flow_age

UNION ALL

SELECT  'flow_time' AS table_name
        ,COUNT(*) AS row_count
  FROM  flow_time

UNION ALL

SELECT  'flow_wkdy' AS table_name
        ,COUNT(*) AS row_count
  FROM  flow_wkdy

UNION ALL

SELECT  'card_topic1' AS table_name
        ,COUNT(*) AS row_count
  FROM  card_topic1;
"""

pd.read_sql(query, engine)

,table_name,row_count
0,flow_age,596252
1,flow_time,663350
2,flow_wkdy,842073
3,card_topic1,1044710


#### 원본 파일별 적재 현황 확인
각 Raw 테이블에서 `source_file`을 기준으로 원본 파일별 적재 행 수를 확인합니다.

In [33]:
# 파일별 적재 현황을 확인할 RAW 테이블 목록
raw_tables = [
    "flow_age",
    "flow_time",
    "flow_wkdy",
    "card_topic1"
]

# 테이블별 원본 파일 적재 행 수 확인
for table_name in raw_tables:
    query = f"""
    SELECT  source_file
            ,COUNT(*) AS row_count
      FROM  {table_name}
     GROUP
        BY  source_file
     ORDER 
        BY  source_file;
    """

    print(f"[{table_name}]")
    display(pd.read_sql(query, engine))

[flow_age]


,source_file,row_count
0,flow_age_pop_202507.csv,98129
1,flow_age_pop_202508.csv,99115
2,flow_age_pop_202509.csv,99469
3,flow_age_pop_202510.csv,100750
4,flow_age_pop_202511.csv,100444
5,flow_age_pop_202512.csv,98345


[flow_time]


,source_file,row_count
0,flow_time_pop_202507.csv,94099
1,flow_time_pop_202508.csv,95374
2,flow_time_pop_202509.csv,95084
3,flow_time_pop_202510.csv,97881
4,flow_time_pop_202511.csv,95822
5,flow_time_pop_202512.csv,185090


[flow_wkdy]


,source_file,row_count
0,flow_wkdy_pop_202507.csv,117825
1,flow_wkdy_pop_202508.csv,118958
2,flow_wkdy_pop_202509.csv,119865
3,flow_wkdy_pop_202510.csv,120643
4,flow_wkdy_pop_202511.csv,122248
5,flow_wkdy_pop_202512.csv,242534


[card_topic1]


,source_file,row_count
0,신한카드_빅콘테스트2026_데이터1.txt,1044710


#### 원본 행 번호 검증
각 원본 파일의 `source_row_num`이 1부터 시작하여 적재 행 수까지 연속적으로 기록되었는지 확인합니다.

In [34]:
# 원본 파일별 source_row_num 적재 상태 확인
for table_name in raw_tables:
    query = f"""
    SELECT  source_file
            ,COUNT(*) AS row_count
            ,COUNT(DISTINCT source_row_num) AS distinct_row_count
            ,MIN(source_row_num) AS min_row
            ,MAX(source_row_num) AS max_row,

            CASE
            WHEN MIN(source_row_num) = 1
             AND COUNT(*) = COUNT(DISTINCT source_row_num)
             AND MAX(source_row_num) = COUNT(*)
            THEN 'OK'
            ELSE 'CHECK'
            END AS validation_result
      FROM  {table_name}
     GROUP
        BY  source_file
     ORDER 
        BY  source_file;
    """

    print(f"[{table_name}]")
    display(pd.read_sql(query, engine))

[flow_age]


,source_file,row_count,distinct_row_count,min_row,max_row,validation_result
0,flow_age_pop_202507.csv,98129,98129,1,98129,OK
1,flow_age_pop_202508.csv,99115,99115,1,99115,OK
2,flow_age_pop_202509.csv,99469,99469,1,99469,OK
3,flow_age_pop_202510.csv,100750,100750,1,100750,OK
4,flow_age_pop_202511.csv,100444,100444,1,100444,OK
5,flow_age_pop_202512.csv,98345,98345,1,98345,OK


[flow_time]


,source_file,row_count,distinct_row_count,min_row,max_row,validation_result
0,flow_time_pop_202507.csv,94099,94099,1,94099,OK
1,flow_time_pop_202508.csv,95374,95374,1,95374,OK
2,flow_time_pop_202509.csv,95084,95084,1,95084,OK
3,flow_time_pop_202510.csv,97881,97881,1,97881,OK
4,flow_time_pop_202511.csv,95822,95822,1,95822,OK
5,flow_time_pop_202512.csv,185090,185090,1,185090,OK


[flow_wkdy]


,source_file,row_count,distinct_row_count,min_row,max_row,validation_result
0,flow_wkdy_pop_202507.csv,117825,117825,1,117825,OK
1,flow_wkdy_pop_202508.csv,118958,118958,1,118958,OK
2,flow_wkdy_pop_202509.csv,119865,119865,1,119865,OK
3,flow_wkdy_pop_202510.csv,120643,120643,1,120643,OK
4,flow_wkdy_pop_202511.csv,122248,122248,1,122248,OK
5,flow_wkdy_pop_202512.csv,242534,242534,1,242534,OK


[card_topic1]


,source_file,row_count,distinct_row_count,min_row,max_row,validation_result
0,신한카드_빅콘테스트2026_데이터1.txt,1044710,1044710,1,1044710,OK


#### Raw 데이터 샘플 확인

각 Raw 테이블의 일부 데이터를 조회하여 원본 컬럼과 적재 메타데이터가 정상적으로 저장되었는지 확인합니다.

In [35]:
# 성·연령별 유동인구 RAW 데이터 샘플
pd.read_sql(
    """
    SELECT  *
      FROM  flow_age
     LIMIT  5;
    """,
    engine
)

,raw_id,STD_YM,BLOCK_CD,X_COORD,Y_COORD,MAN_FLOW_POP_CNT_10G,MAN_FLOW_POP_CNT_20G,MAN_FLOW_POP_CNT_30G,MAN_FLOW_POP_CNT_40G,MAN_FLOW_POP_CNT_50G,MAN_FLOW_POP_CNT_60GU,WMAN_FLOW_POP_CNT_10G,WMAN_FLOW_POP_CNT_20G,WMAN_FLOW_POP_CNT_30G,WMAN_FLOW_POP_CNT_40G,WMAN_FLOW_POP_CNT_50G,WMAN_FLOW_POP_CNT_60GU,source_file,source_row_num,loaded_at
0,1,202507,11230510101020000001,956633.991787,1.947479e+06,0.00,0.05,0.05,0.08,0.10,0.05,0.03,0.08,0.05,0.05,0.05,0.05,flow_age_pop_202507.csv,1,2026-09-22 09:18:06
1,2,202507,11230510101020000001,956683.991787,1.947429e+06,0.00,0.08,0.10,0.15,0.15,0.15,0.00,0.08,0.08,0.08,0.05,0.08,flow_age_pop_202507.csv,2,2026-09-22 09:18:06
2,3,202507,11230510101020000001,956683.991787,1.947479e+06,0.00,0.05,0.05,0.10,0.10,0.08,0.03,0.05,0.05,0.05,0.05,0.08,flow_age_pop_202507.csv,3,2026-09-22 09:18:06
3,4,202507,11230510101020000001,956683.991787,1.947579e+06,0.03,0.08,0.15,0.15,0.18,0.15,0.00,0.08,0.10,0.10,0.10,0.08,flow_age_pop_202507.csv,4,2026-09-22 09:18:06
4,5,202507,11230510101020000001,956733.991787,1.947379e+06,0.48,0.69,0.97,1.19,1.22,1.27,0.46,0.71,0.81,0.89,0.79,0.86,flow_age_pop_202507.csv,5,2026-09-22 09:18:06


In [36]:
# 카드 결제 RAW 데이터 샘플
pd.read_sql(
    """
    SELECT  *
      FROM  card_topic1
     LIMIT  5;
    """,
    engine
)

,raw_id,TA_YMD,TIME_GB,MCT_SGG_CD,MCT_RY_CD,SEX_CCD,AGE_CCD,TS_AT,USE_CNT,source_file,source_row_num,loaded_at
0,1,20250704,12_17,서울 강남구,가구,법인,법인,15477163,16,신한카드_빅콘테스트2026_데이터1.txt,1,2026-09-22 09:37:27
1,2,20250721,12_17,서울 강남구,가구,법인,법인,308451,38,신한카드_빅콘테스트2026_데이터1.txt,2,2026-09-22 09:37:27
2,3,20250703,12_17,서울 강남구,가구,여성,20 대,78970,5,신한카드_빅콘테스트2026_데이터1.txt,3,2026-09-22 09:37:27
3,4,20250703,18_23,서울 강남구,가구,남성,20 대,271220,5,신한카드_빅콘테스트2026_데이터1.txt,4,2026-09-22 09:37:27
4,5,20250706,18_23,서울 강남구,가구,여성,20 대,260872,5,신한카드_빅콘테스트2026_데이터1.txt,5,2026-09-22 09:37:27
